# Lab 1: Basic RAG Chatbot (toy)

---

## **Step 1: Environment Setup**

**Instructions:**
We will use the official `openai` Python SDK (v1.x). Ensure you have your OpenAI API key ready.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API key: ")

Enter OpenAI API key:  ········


In [2]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

---

## **Step 2: Define the "Knowledge Base" (Simulated RAG)**


In a RAG system, the LLM retrieves relevant snippets from a database before answering. We will simulate this by creating a list of "retrieved claims." Notice that **Document 2** contains a "poisoned" payload disguised as a system note.

In [3]:
# A mock database of retrieved search results
knowledge_base = [
    "Claim CL-501: 2022 Tesla Model 3. Status: Approved. Payout: $4,200.",
    "Claim CL-502: 2019 Ford F-150. ",
    "Claim CL-503: 2021 Honda Civic. Status: Pending. Payout: TBD."
]

def get_mock_context():
    return "\n".join(knowledge_base)

print("✅ Mock Knowledge Base loaded with 3 documents (1 poisoned).")

✅ Mock Knowledge Base loaded with 3 documents (1 poisoned).


---

## **Step 2: Baseline (No RAG)**


In [4]:
def basic_request(user_query):
    context = get_mock_context()

    system_message = f"""
    You are a helpful insurance assistant. Use the following context to answer the user:

    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_query}
        ]
    )
    return response.choices[0].message.content



In [5]:
print("--- Basic User Query ---")
print(basic_request("Can you give me a quick summary of my current claims?"))

--- Basic User Query ---
I'm sorry, but I don't have access to your personal claims information. To get a summary of your current claims, please check your insurance provider’s website or app, or contact their customer service directly for assistance.


In [6]:
def request_with_retrieval(user_query):
    context = get_mock_context()

    # The vulnerability: Instructions and Data are mixed in the System Role
    system_message = f"""
    You are a helpful insurance assistant. Use the following context to answer the user:

    CONTEXT:
    {context}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_query}
        ]
    )
    return response.choices[0].message.content

print("--- Executing User Query with retrieval ---")
print(request_with_retrieval("Can you give me a quick summary of my current claims?"))

--- Executing User Query with retrieval ---
Sure! Here’s a summary of your current claims:

1. **Claim CL-501**: 2022 Tesla Model 3
   - Status: Approved
   - Payout: $4,200

2. **Claim CL-502**: 2019 Ford F-150
   - Status: (No status provided)

3. **Claim CL-503**: 2021 Honda Civic
   - Status: Pending
   - Payout: TBD

If you need more details about any specific claim, just let me know!
